In [1]:
# Alexnet, VGG-19, ResNet-18
from torchvision import models

from torchvision.models import alexnet, vgg19, resnet18

alexnet_model = alexnet(weights = models.AlexNet_Weights.DEFAULT)
vgg19_model = vgg19(weights = models. VGG19_Weights.DEFAULT)
resnet18_model = resnet18(weights = models.ResNet18_Weights.DEFAULT)

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/badmazzz/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/2


In [3]:
import os
import torch
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

img_size = 224

train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),  # Fixed capitalization
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],     # Fixed 8.456 -> 0.456
        std=[0.229, 0.224, 0.225],
    )
])

test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
])

path = "/Users/badmazzz/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/2"  # Replace with your dataset root path
train_dir = os.path.join(path, "Training")
test_dir = os.path.join(path, "Testing")

train_base = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_base   = datasets.ImageFolder(root=train_dir, transform=test_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

print("Classes:", train_base.classes)
print("Class to index:", train_base.class_to_idx)

val_ratio = 0.2
n_val = int(len(train_base) * val_ratio)
n_train = len(train_base) - n_val

indices = list(range(len(train_base)))
train_indices, val_indices = random_split(
    indices,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_dataset = Subset(train_base, train_indices)
val_dataset   = Subset(val_base, val_indices)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

images, labels = next(iter(train_loader))
print("Train batch shape:", images.shape)
print("Train labels shape:", labels.shape)

Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Class to index: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
Train batch shape: torch.Size([32, 3, 224, 224])
Train labels shape: torch.Size([32])


In [4]:
train_loader, val_loader, test_loader
num_classes = 4 # glioma, meningioma, notumor, pituitary

In [5]:
import torch
import torch.nn as nn 
import torch.optim as optim 
from torchvision import models 
import matplotlib. pyplot as plt

device = torch. device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [6]:
# Define simple MLP and CNN

class SimpleMLP (nn.Module) :
    def __init__(self, num_classes=4, img_size=224):
        super().__init__()
        # assuming 3-channel images resized to (img_size, img_size)
        in_features = 3 * img_size * img_size
        self.net = nn.Sequential(
            nn. Flatten(), 
            nn. Linear (in_features, 512),
            nn. ReLU(), 
            nn. Linear(512, 128), 
            nn. ReLU(), 
            nn. Linear(128, num_classes)
        )

    def forward (self, x):
        return self.net(x)
    
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__() 
        self.features = nn.Sequential(
            nn. Conv2d(3, 32, kernel_size=3, padding=1), 
            nn. ReLU(),
            nn.MaxPool2d(2), # 112x112
            nn.Conv2d(32, 64, kernel_size=3, padding=1), 
            nn. ReLU(),
            nn.MaxPool2d(2), # 56x56
            nn. Conv2d(64, 128, kernel_size=3, padding=1), 
            nn. ReLU(), 
            nn. MaxPool2d(2), # 28x28
        )
        self.classifier = nn.Sequential(
            nn. Flatten(),
            nn. Linear (128 * 28 * 28, 256), 
            nn.ReLU(), 
            nn. Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [7]:
# Helpers to get pretrained models and replace head

def get_alexnet (num_classes) :
    model = models.alexnet(pretrained=True)
    in_features = model.classifier[6]. in_features
    model. classifier[6] = nn.Linear(in_features, num_classes)
    return model

def get_vgg16 (num_classes):
    model = models.vgg16(pretrained=True)
    in_features = model. classifier[6]. in_features
    model.classifier[6] = nn.Linear(in_features, num_classes)
    return model

def get_convnext_tiny(num_classes) :
    model = models. convnext_tiny(pretrained=True)
    in_features = model.classifier[2]. in_features
    model. classifier[2] = nn.Linear(in_features, num_classes)
    return model

def get_vit_b16 (num_classes) :
    model = models. vit_b_16(pretrained=True)
    in_features = model.heads.head. in_features
    model. heads.head = nn. Linear(in_features, num_classes)
    return model

def get_resnet18(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


In [8]:
# 4. Instantiate all models

num_classes = 4
# make sure this matches your dataset
models_dict ={
    "MLP": SimpleMLP(num_classes=num_classes, img_size=224),
    "SimpleCNN": SimpleCNN(num_classes=num_classes),
    "AlexNet": get_alexnet (num_classes),
    "VGG16": get_vgg16 (num_classes),
    # "ResNet18": get_resnet18(num_classes),
    "ConvNeXt Tiny": get_convnext_tiny(num_classes),
    "ViT-B16": get_vit_b16(num_classes)
}

num_epochs = 2 # for demo
lr = 1e-4 # single LR for all, for simplicity

all_train_hist = {}
all_val_hist = {}

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weight

In [9]:
# Generic Training and Validation Function

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5, device=device):

    model = model.to(device)
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_train_loss = running_loss / total
        epoch_train_acc = correct / total

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total

        history["train_loss"].append(epoch_train_loss)
        history["train_acc"].append(epoch_train_acc)
        history["val_loss"].append(epoch_val_loss)
        history["val_acc"].append(epoch_val_acc)

        print(
            f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | "
            f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc * 100:.2f}% | "
            f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc * 100:.2f}%"
        )

    return history


In [ ]:
# 5. Training Loop across all models in models_dict

criterion = nn.CrossEntropyLoss()

for model_name, model in models_dict.items():
    print(f"\n" + "=" * 25 + f" Training: {model_name} " + "=" * 25)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Train and evaluate
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=num_epochs,
        device=device
    )
    
    # Store results
    all_train_hist[model_name] = history



========================= Training: MLP =========================
Epoch [01/02] | Train Loss: 1.0346 - Train Acc: 62.83% | Val Loss: 0.6901 - Val Acc: 74.91%
Epoch [02/02] | Train Loss: 0.7017 - Train Acc: 73.48% | Val Loss: 0.8741 - Val Acc: 68.48%

========================= Training: SimpleCNN =========================
Epoch [01/02] | Train Loss: 0.7223 - Train Acc: 70.51% | Val Loss: 0.6378 - Val Acc: 73.12%
Epoch [02/02] | Train Loss: 0.4520 - Train Acc: 82.83% | Val Loss: 0.4369 - Val Acc: 83.57%

========================= Training: AlexNet =========================
Epoch [01/02] | Train Loss: 0.4053 - Train Acc: 85.00% | Val Loss: 0.2200 - Val Acc: 91.34%
Epoch [02/02] | Train Loss: 0.1971 - Train Acc: 93.08% | Val Loss: 0.2052 - Val Acc: 91.70%

========================= Training: VGG16 =========================
Epoch [01/02] | Train Loss: 0.4143 - Train Acc: 84.40% | Val Loss: 0.2720 - Val Acc: 89.38%
